# S0 — 도구·바닥 (Tooling & Noise Floor)

**목적:** 본실험 전 엔진을 검증하고 노이즈 바닥을 먼저 잰다. 이게 통과해야 S1(척추)이 의미를 가짐.

**성공 기준**
1. `roundtrip_test(8)` 통과 + W8 PTQ ≈ 무손실 → manual conv 양자화 정확
2. `ptq()` silent-skip assert 통과 → 모든 대상 층이 *실제로* 양자화됨
3. **W4 PTQ 갭 > 노이즈 바닥** → 회복할 게 있다
4. HVP 셀 finite·동작 + δᵀHδ 부호 로깅

엔진(`qat_engine.py`)이 모든 로직. 이 노트북은 호출·점검만 한다.

In [ ]:
# --- Colab 셋업: repo clone/pull + 의존성 ---
import os
REPO = '/content/26_Capstone'                 # 절대경로 → 재실행해도 중첩 clone 안 됨 (Codex)
if not os.path.isdir(REPO):
    !git clone -q https://github.com/u-nsiq/26_Capstone.git {REPO}
else:
    !git -C {REPO} pull -q                     # 내가 push한 엔진 수정을 여기서 당겨옴
os.chdir(REPO)
!pip install -q -r requirements.txt
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# --- 엔진 import + Drive(영구저장) + 경로 ---
from qat_engine import *
import numpy as np

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ART = '/content/drive/MyDrive/26_Capstone'
except Exception:
    ART = './_local_art'   # 로컬 폴백
os.makedirs(f'{ART}/checkpoints', exist_ok=True)
os.makedirs(f'{ART}/outputs', exist_ok=True)
DATA_ROOT = f'{ART}/data'
CKPT      = f'{ART}/checkpoints/resnet18_cifar100_fp32.pt'
OUT       = f'{ART}/outputs/runs.jsonl'
print('device', DEVICE, '| ART', ART)

In [ ]:
# --- 잠긴 설정 (05/06) ---
BITS_MAIN       = 4      # 주력 W4 weight-only
BASELINE_EPOCHS = 60     # 첫 파이프라인 점검만이면 15~20으로 빠르게; 본실험 전 60+로 제대로(캐시됨)
SHORT_STEPS     = 100    # S0 노이즈바닥용 짧은 회복 길이
LR              = 1e-3   # 10분 sanity 후 고정
MOMENTUM        = 0.0    # vanilla GD (#3) — 진단/핵심 run
N_SEED_NOISE    = 5      # 노이즈바닥 반복 횟수

## Step 0–1 — FP32 baseline (천장) 확보
캐시 있으면 즉시 로드, 없으면 한 번 학습→Drive 캐시. 잘 학습된 FP32라야 §4.1 선형화(최소점 근방)가 깨끗 = 이론 위생.

In [ ]:
train_loader, val_loader, calib_loader = get_loaders('cifar100', batch=128, calib_size=512, data_root=DATA_ROOT)
model_fp = load_model('resnet18', 'cifar100', ckpt=CKPT if os.path.exists(CKPT) else None)
if not os.path.exists(CKPT):
    model_fp, _ = train_baseline(model_fp, train_loader, val_loader, epochs=BASELINE_EPOCHS, ckpt_path=CKPT)
fp_acc = evaluate(model_fp, val_loader)
print(f'FP32 천장 top1 = {fp_acc:.2f}')

## Step 2 — W8 sanity (round-trip + 무손실)
conv는 torchao로 검증 불가(P0B) → round-trip 단위테스트 + W8 PTQ가 거의 무손실인지로 엔진 정확성 보증.

In [ ]:
# (a) round-trip 단위테스트  (b) W8 PTQ가 거의 무손실인가
print('round-trip W8:', roundtrip_test(8))
m8 = ptq(load_model('resnet18','cifar100',ckpt=CKPT), n_bits=8)   # silent-skip assert 내장
acc8 = evaluate(m8, val_loader)
print(f'W8 PTQ top1 = {acc8:.2f}  (FP32 {fp_acc:.2f}, gap {fp_acc-acc8:.2f}%p -> 무손실 기대)')
assert abs(fp_acc - acc8) < 1.5, f'W8가 무손실 아님 (gap {fp_acc-acc8:.2f}%p) — 양자화기 점검 (Codex)'
print('W8 sanity OK (거의 무손실)')

## Step 3 — W4 PTQ 갭 (회복 대상)

In [ ]:
m4 = ptq(load_model('resnet18','cifar100',ckpt=CKPT), n_bits=BITS_MAIN)
acc4 = evaluate(m4, val_loader)
W4_GAP = fp_acc - acc4
print(f'W4 PTQ top1 = {acc4:.2f}  -> gap {W4_GAP:.2f}%p')

## Step 4 — 노이즈 바닥 (단일층 recovery의 run-to-run std)
측정하려는 건 *회복 측정 전체*의 변동(eval 분산 아님). 대표 층 하나를 짧게 회복 → seed만 바꿔 N회 → recovery std. (#7)

In [ ]:
qnames = [n for n, m in m4.named_modules() if isinstance(m, (QConv2d, QLinear))]
REP_LAYER = qnames[len(qnames)//2]      # 중간 대표 층
print('대표 층:', REP_LAYER, f'(총 {len(qnames)} 양자화 층)')

def one_layer_recovery(seed):
    m = ptq(load_model('resnet18','cifar100',ckpt=CKPT), n_bits=BITS_MAIN)
    base = evaluate(m, val_loader)
    set_trainable(m, [REP_LAYER])
    r = short_qat(m, train_loader, val_loader, steps=SHORT_STEPS,
                  lr=LR, momentum=MOMENTUM, seed=seed, eval_at=(SHORT_STEPS,))
    return r[SHORT_STEPS] - base       # recovery(%p)

nf_mean, nf_std, nf_vals = noise_floor(one_layer_recovery, n=N_SEED_NOISE)
print(f'노이즈 바닥: recovery {nf_mean:.2f} +/- {nf_std:.2f} %p   vals={[round(v,2) for v in nf_vals]}')

## Step 5 — 게이트: W4 갭 > 노이즈 바닥?

In [ ]:
ratio = W4_GAP / max(nf_std, 1e-6)
print(f'W4 gap {W4_GAP:.2f}%p  vs  noise std {nf_std:.2f}%p  ->  {ratio:.1f}x')
assert W4_GAP > 5*nf_std, 'gap이 노이즈 대비 작음 — 회복 신호가 묻힐 수 있음(baseline/비트 재검토)'
print('게이트 통과 OK  (회복할 갭이 노이즈보다 충분히 큼)')

## Step 6 — HVP smoke (finite + δᵀHδ 부호)
검증된 Pearlmutter HVP가 실제 W4 모델에서 finite·동작하는지. δᵀHδ<0이면 버그가 아니라 breakdown 신호(진짜 H는 PTQ점서 PSD 아님) — S1에서 다룸. (#5)

In [ ]:
deltas = quant_error(m4)
prox = hvp_proxies(m4, REP_LAYER, deltas[REP_LAYER], calib_loader, n_batches=4)
dt, nh, sg = prox['dtHd'], prox['normHd2'], prox['sign']
print(f'{REP_LAYER}:  dtHd={dt:.4e} (sign {sg:+d})   normHd2={nh:.4e}')
assert np.isfinite(dt) and np.isfinite(nh) and nh > 0, 'HVP non-finite/zero — 점검 (Codex: finite·non-zero)'
print('HVP 동작 OK   (음수 sign이면 버그 아니라 breakdown 신호)')

In [ ]:
# --- S0 요약을 Drive에 영구 기록 (repo 아님, Drive의 runs.jsonl) ---
s0_summary = dict(
    fp_acc=fp_acc, w8_acc=acc8, w8_gap=fp_acc-acc8,
    w4_acc=acc4, w4_gap=W4_GAP,
    noise_mean=nf_mean, noise_std=nf_std,
    rep_layer=REP_LAYER, hvp_dtHd=dt, hvp_sign=sg, hvp_normHd2=nh,
)
log_run({'phase': 'S0', 'model': 'resnet18', 'dataset': 'cifar100', 'bits': BITS_MAIN}, s0_summary, path=OUT)
print('S0 요약 저장 →', OUT)
print(s0_summary)

## S0 완료 → 다음 S1
통과하면: 엔진 검증됨 · 노이즈 바닥 확보 · W4 갭 게이트 통과 · HVP 동작.

**다음:** `S1_diagnostic_inversion.ipynb` — 단일층 R_l(t) 궤적으로 **역전(단기≠수렴)** + proxy 예측(δᵀHδ·‖Hδ‖²) 검증.